Import Necessary Libraries

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
import re
import numpy as np
from gensim.models import Word2Vec
from tqdm import tqdm
import requests

/Users/sandhyas/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


GITHUB Details - Replace GITHUB ACCESS KEY with your own Personal access token

In [ ]:
github_token = "GITHUB ACCESS KEY"
headers = {"Authorization": f"token {github_token}"}

Function for Text pre-processing

In [3]:
def clean_text(text):
    text = re.sub(r'`.*?`', '', text)
    text = re.sub(r'[\\U00010000-\\U0010ffff]', '', text)
    text = re.sub(r'[\\W]+', ' ', text)
    return text.lower().strip()

# Tokenize text for Word2Vec
def tokenize_text(text):
    return text.split()

# Convert text to vector by averaging word vectors
def text_to_vector(model, tokens, vector_size):
    vec = np.zeros(vector_size)
    count = 0
    for word in tokens:
        if word in model.wv:
            vec += model.wv[word]
            count += 1
    return vec / count if count > 0 else vec

Discussion to Issues

In [4]:
df = pd.read_csv("../Dataset/DiscussionToIssue.csv")

In [5]:
encoder = LabelEncoder()
df['IsIssueRaised'] = encoder.fit_transform(df['IsIssueRaised'])
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)

Training ...

In [6]:
df_shuffled['concatenated'] = (df_shuffled['Title'] + ' ' + df_shuffled['Description'] + ' ' + df_shuffled['Comments']).apply(clean_text)

df_shuffled['tokens'] = df_shuffled['concatenated'].apply(tokenize_text)
w2v_model = Word2Vec(sentences=df_shuffled['tokens'], vector_size=100, window=5, min_count=1, workers=4, seed=42)

# Convert all texts to vectors
X = np.array([text_to_vector(w2v_model, tokens, 100) for tokens in df_shuffled['tokens']])
y = df_shuffled['IsIssueRaised'].values


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [8]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

Testing ...

In [9]:
y_pred = model.predict(X_test)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.713963963963964

Classification Report:
               precision    recall  f1-score   support

           0       0.67      0.69      0.68       196
           1       0.75      0.73      0.74       248

    accuracy                           0.71       444
   macro avg       0.71      0.71      0.71       444
weighted avg       0.72      0.71      0.71       444



Discussion to Issues with Description Alone

In [10]:
df_shuffled['concatenated'] = (df_shuffled['Title'] + ' ' + df_shuffled['Description']).apply(clean_text)

df_shuffled['tokens'] = df_shuffled['concatenated'].apply(tokenize_text)
w2v_model = Word2Vec(sentences=df_shuffled['tokens'], vector_size=100, window=5, min_count=1, workers=4, seed=42)

# Convert all texts to vectors
X = np.array([text_to_vector(w2v_model, tokens, 100) for tokens in df_shuffled['tokens']])
y = df_shuffled['IsIssueRaised'].values

Training ...

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [12]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

Testing ...

In [13]:
y_pred = model.predict(X_test)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.6036036036036037

Classification Report:
               precision    recall  f1-score   support

           0       0.55      0.54      0.55       196
           1       0.64      0.65      0.65       248

    accuracy                           0.60       444
   macro avg       0.60      0.60      0.60       444
weighted avg       0.60      0.60      0.60       444



Discussion to Issues with First Comment alone

Title + Description + First Comment

Function to extract Repository and Discussion number

In [14]:
def extract_github_path(url):
    match = re.search(r'github\.com/([^?#]*)', url)
    return match.group(1) if match else None

Fetch the First comment from Discussion 

In [17]:
df_shuffled['Comment'] =  None
for index,row in tqdm(df_shuffled.iterrows()):
  repo = extract_github_path(row['Issue'])
  curl = f'https://api.github.com/repos/{repo}/comments'
  repo_comment=[]
  cresponse = requests.get(curl,  headers=headers)
  if cresponse.status_code == 200:
    issueComments = cresponse.json()
    cnt=0
    for comment in issueComments:
        repo_comment.append(comment['body'])
        cnt+=1
        if(cnt<1):
          break
  df_shuffled.at[index, 'Comment'] = repo_comment

2218it [15:36,  2.37it/s]


Training ...

In [18]:
df_shuffled['concatenated'] = (df_shuffled['Title'] + ' ' + df_shuffled['Description']+' '+str(df_shuffled['Comment'])).apply(clean_text)
df_shuffled['tokens'] = df_shuffled['concatenated'].apply(tokenize_text)
w2v_model = Word2Vec(sentences=df_shuffled['tokens'], vector_size=100, window=5, min_count=1, workers=4, seed=42)

# Convert all texts to vectors
X = np.array([text_to_vector(w2v_model, tokens, 100) for tokens in df_shuffled['tokens']])
y = df_shuffled['IsIssueRaised'].values

In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [20]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

Testing ...

In [21]:
y_pred = model.predict(X_test)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.6418918918918919

Classification Report:
               precision    recall  f1-score   support

           0       0.60      0.57      0.58       196
           1       0.67      0.70      0.69       248

    accuracy                           0.64       444
   macro avg       0.64      0.63      0.64       444
weighted avg       0.64      0.64      0.64       444



Issues to Discussion

In [22]:
df1 = pd.read_csv("../Dataset/IssueToDiscussion.csv")

In [23]:
encoder = LabelEncoder()
df1['ConvertedFromIssue'] = encoder.fit_transform(df1['ConvertedFromIssue'])
df1_shuffled = df1.sample(frac=1, random_state=42).reset_index(drop=True)

Training ...

In [24]:
df1_shuffled['concatenated'] = (df1_shuffled['Title'] + ' ' + df1_shuffled['Description'] + ' ' + df1_shuffled['Comments']).apply(clean_text)

df1_shuffled['tokens'] = df1_shuffled['concatenated'].apply(tokenize_text)
w2v_model = Word2Vec(sentences=df1_shuffled['tokens'], vector_size=100, window=5, min_count=1, workers=4, seed=42)

# Convert all texts to vectors
X = np.array([text_to_vector(w2v_model, tokens, 100) for tokens in df1_shuffled['tokens']])
y = df1_shuffled['ConvertedFromIssue'].values


In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [26]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

Testing ...

In [27]:
y_pred = model.predict(X_test)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.6940298507462687

Classification Report:
               precision    recall  f1-score   support

           0       0.70      0.72      0.71       209
           1       0.69      0.67      0.68       193

    accuracy                           0.69       402
   macro avg       0.69      0.69      0.69       402
weighted avg       0.69      0.69      0.69       402



Issue to Discussion with Description alone

In [28]:
df1 = pd.read_csv("../Dataset/IssueToDiscussion.csv")

In [29]:
encoder = LabelEncoder()
df1['ConvertedFromIssue'] = encoder.fit_transform(df1['ConvertedFromIssue'])
df1_shuffled = df1.sample(frac=1, random_state=42).reset_index(drop=True)

Training ...

In [30]:
df1_shuffled['concatenated'] = (df1_shuffled['Description']).apply(clean_text)

df1_shuffled['tokens'] = df1_shuffled['concatenated'].apply(tokenize_text)
w2v_model = Word2Vec(sentences=df1_shuffled['tokens'], vector_size=100, window=5, min_count=1, workers=4, seed=42)

# Convert all texts to vectors
X = np.array([text_to_vector(w2v_model, tokens, 100) for tokens in df1_shuffled['tokens']])
y = df1_shuffled['ConvertedFromIssue'].values


In [31]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [32]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

Testing ...

In [33]:
y_pred = model.predict(X_test)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.7238805970149254

Classification Report:
               precision    recall  f1-score   support

           0       0.73      0.74      0.74       209
           1       0.72      0.70      0.71       193

    accuracy                           0.72       402
   macro avg       0.72      0.72      0.72       402
weighted avg       0.72      0.72      0.72       402



Issue to Discussion with Description + First Comment

In [34]:
df1 = pd.read_csv("../Dataset/IssueToDiscussion.csv")

In [35]:
encoder = LabelEncoder()
df1['ConvertedFromIssue'] = encoder.fit_transform(df1['ConvertedFromIssue'])
df1_shuffled = df1.sample(frac=1, random_state=42).reset_index(drop=True)

Function to extract Repository and Issue number

In [36]:
def extract_github_path(url):
    match = re.search(r'github\.com/([^?#]*)', url)
    return match.group(1) if match else None

Download the First Comment of an Issue

In [38]:
df1_shuffled['Comment'] =  None
for index,row in tqdm(df1_shuffled.iterrows()):
  repo = extract_github_path(row['Issue'])
  curl = f'https://api.github.com/repos/{repo}/comments'
  repo_comment=[]
  cresponse = requests.get(curl,  headers=headers)
  if cresponse.status_code == 200:
    issueComments = cresponse.json()
    cnt=0
    for comment in issueComments:
        repo_comment.append(comment['body'])
        cnt+=1
        if(cnt<1):
          break
  df1_shuffled.at[index, 'Comment'] = repo_comment

2010it [13:45,  2.43it/s]


Training ...

In [39]:
df1_shuffled['concatenated'] = (df1_shuffled['Description'] + ' ' + str(df1_shuffled['Comment'])).apply(clean_text)

df1_shuffled['tokens'] = df1_shuffled['concatenated'].apply(tokenize_text)
w2v_model = Word2Vec(sentences=df1_shuffled['tokens'], vector_size=100, window=5, min_count=1, workers=4, seed=42)

# Convert all texts to vectors
X = np.array([text_to_vector(w2v_model, tokens, 100) for tokens in df1_shuffled['tokens']])
y = df1_shuffled['ConvertedFromIssue'].values


In [40]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [41]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

Testing ...

In [42]:
y_pred = model.predict(X_test)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.6890547263681592

Classification Report:
               precision    recall  f1-score   support

           0       0.70      0.69      0.70       209
           1       0.67      0.68      0.68       193

    accuracy                           0.69       402
   macro avg       0.69      0.69      0.69       402
weighted avg       0.69      0.69      0.69       402



Issue to Discussion with Comments

In [43]:
df1 = pd.read_csv("../Dataset/IssueToDiscussion.csv")

In [44]:
encoder = LabelEncoder()
df1['ConvertedFromIssue'] = encoder.fit_transform(df1['ConvertedFromIssue'])
df1_shuffled = df1.sample(frac=1, random_state=42).reset_index(drop=True)

Training ...

In [45]:
df1_shuffled['concatenated'] = (df1_shuffled['Comments']).apply(clean_text)

df1_shuffled['tokens'] = df1_shuffled['concatenated'].apply(tokenize_text)
w2v_model = Word2Vec(sentences=df1_shuffled['tokens'], vector_size=100, window=5, min_count=1, workers=4, seed=42)

# Convert all texts to vectors
X = np.array([text_to_vector(w2v_model, tokens, 100) for tokens in df1_shuffled['tokens']])
y = df1_shuffled['ConvertedFromIssue'].values


In [46]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [47]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

Testing ...

In [48]:
y_pred = model.predict(X_test)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.582089552238806

Classification Report:
               precision    recall  f1-score   support

           0       0.60      0.60      0.60       209
           1       0.57      0.56      0.56       193

    accuracy                           0.58       402
   macro avg       0.58      0.58      0.58       402
weighted avg       0.58      0.58      0.58       402

